# Supervised LSTM Autoencoder

This notebook keeps the autoencoder representation-learning idea from `baseline_autoencoder.ipynb`, but makes the encoder supervised and sequential. The model trains an LSTM encoder with reconstruction, return prediction, alpha prediction, and date-aware rank losses, then builds a weekly long-only portfolio from `expected_alpha / historical_vol_20d`.


In [1]:
# Bootstrap: works locally, in Colab, or when repo_root is set manually.
import os
import sys
import subprocess
import tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_REPO_URL = "https://github.com/halloyy/Portfolio-Optimization-Lib.git"


def _is_repo_root(path: Path) -> bool:
    path = Path(path).expanduser()
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def _candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    if "repo_root" in globals():
        candidates.append(Path(repo_root).expanduser())
    if os.environ.get("PORTFOLIO_REPO_ROOT"):
        candidates.append(Path(os.environ["PORTFOLIO_REPO_ROOT"]).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    candidates.extend(
        [
            Path("/content/Portfolio-Optimization-Lib"),
            Path("/content/Portfolio-Optimizer"),
            Path("/workspace/Portfolio-Optimization-Lib"),
            Path("/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib"),
        ]
    )
    return candidates


def _find_repo_root() -> Path | None:
    for candidate in _candidate_roots():
        if _is_repo_root(candidate):
            return candidate.resolve()
    return None


repo_root = _find_repo_root()

if repo_root is None and IN_COLAB:
    repo_url = os.environ.get("PORTFOLIO_REPO_URL", DEFAULT_REPO_URL)
    clone_target = Path(os.environ.get("PORTFOLIO_REPO_ROOT", "/content/Portfolio-Optimization-Lib")).expanduser()
    if not clone_target.exists():
        subprocess.run(["git", "clone", repo_url, str(clone_target)], check=True)
    repo_root = clone_target.resolve() if _is_repo_root(clone_target) else None
    if repo_root is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)], check=True)

if repo_root is None:
    raise RuntimeError(
        "Cannot find repo root. Run this notebook from inside Portfolio-Optimization-Lib, "
        "or set repo_root = Path('/path/to/Portfolio-Optimization-Lib') before this cell."
    )

os.chdir(repo_root)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio_matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)
print("python    =", sys.executable)


repo_root = /content/Portfolio-Optimization-Lib
python    = /usr/bin/python3


In [2]:
import json
import math
from dataclasses import replace
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)

warnings.filterwarnings("ignore")
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


torch: 2.10.0+cu128 | cuda: True


## Configuration

The defaults are intentionally focused: `shared_set_2`, a 5-trading-day target, and weekly portfolio updates. The epoch count and MLflow logging flag can be overridden from the environment for fast smoke runs.


In [3]:
DATASET_NAME = "shared_set_1"
MODEL_NAME = "hannah_supervised_lstm_autoencoder"
RUN_NAME = "Hannah_Supervised_LSTM_AE"
HORIZON = 5
SEQ_LEN = 20
REBALANCE_FREQUENCY = "weekly"

LATENT_DIM = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = int(os.environ.get("LSTM_AE_EPOCHS", "18"))
DATES_PER_BATCH = 16
PATIENCE = 5
MASK_RATE = 0.10
GRAD_CLIP = 1.0

RECON_WEIGHT = 1.00
ALPHA_WEIGHT = 2.00
RETURN_WEIGHT = 0.50
RANK_WEIGHT = 0.20

TOP_FRACTION = 0.50
MAX_WEIGHT = 0.10
TURNOVER_BLEND = 0.50
RANDOM_SEED = 99
LOG_TO_MLFLOW = os.environ.get("SKIP_MLFLOW", "0") != "1"

output_dir = repo_root / "runs" / "supervised_lstm_autoencoder"
output_dir.mkdir(parents=True, exist_ok=True)
NOTEBOOK_PATH = repo_root / "MODELS" / "Hannah" / "supervised_lstm_autoencoder.ipynb"

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
tradable_tickers = [ticker for ticker in spec.tickers if ticker != spec.benchmark_ticker]
# shared_set_2 includes SPY in spec.tickers; this copy keeps SPY as benchmark but removes it from tradable weights.
backtest_spec = replace(spec, tickers=tradable_tickers, dataset_id=DATASET_NAME)
splits = split_dates(DATASET_NAME, repo_root=repo_root)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True

print("Dataset preset:", DATASET_NAME)
print("Dataset display name:", spec.name)
print("Tickers in preset:", len(spec.tickers))
print("Tradable tickers:", len(tradable_tickers))
print("Benchmark ticker:", spec.benchmark_ticker)
print("Train/Val/Test windows:", splits)
print("Epochs:", EPOCHS, "| MLflow logging:", LOG_TO_MLFLOW)


Dataset preset: shared_set_1
Dataset display name: sp500_full_universe
Tickers in preset: 503
Tradable tickers: 503
Benchmark ticker: SPY
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}
Epochs: 18 | MLflow logging: True


## Features

`build_model_features` is the reusable inference-safe feature builder. It uses only raw price history, never forward targets, and adds regime, longer-horizon trend/risk, and breadth features on top of the shared toolkit catalog.



## Optional: Tailscale For Colab MLflow

Run this cell only when the notebook kernel is a Colab runtime and you want MLflow logging to reach the team Tailscale host. The runtime must have a Tailscale auth key available as `TS_AUTHKEY` in the environment or Colab Secrets. Do not paste auth keys directly into the notebook.


In [4]:

def setup_tailscale_for_colab(hostname: str = "colab-mlflow", proxy_port: int = 1055) -> bool:
    """Install/start Tailscale userspace networking in a Colab runtime.

    Returns True when the local HTTP/SOCKS proxy is configured for this kernel.
    Requires TS_AUTHKEY to be set in the environment or Colab Secrets.
    """
    import os
    import shutil
    import subprocess
    import sys
    import time

    try:
        from google.colab import userdata  # type: ignore
        if not os.environ.get("TS_AUTHKEY"):
            secret = userdata.get("TS_AUTHKEY")
            if secret:
                os.environ["TS_AUTHKEY"] = secret
    except Exception:
        pass

    auth_key = os.environ.get("TS_AUTHKEY")
    if not auth_key:
        print("Tailscale setup skipped: set TS_AUTHKEY in Colab Secrets or os.environ first.")
        return False

    if shutil.which("tailscale") is None:
        print("Installing Tailscale in this runtime...")
        subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True, check=True)

    tailscaled_cmd = [
        "tailscaled",
        "--tun=userspace-networking",
        f"--socks5-server=127.0.0.1:{proxy_port}",
        f"--outbound-http-proxy-listen=127.0.0.1:{proxy_port}",
        "--state=mem:",
    ]
    print("Starting tailscaled userspace proxy...")
    subprocess.Popen(
        tailscaled_cmd,
        stdout=open("/tmp/tailscaled.log", "ab"),
        stderr=subprocess.STDOUT,
    )
    time.sleep(3)

    subprocess.run(
        ["tailscale", "up", "--auth-key", auth_key, "--hostname", hostname, "--reset"],
        check=True,
    )

    proxy = f"http://127.0.0.1:{proxy_port}"
    os.environ["HTTP_PROXY"] = proxy
    os.environ["HTTPS_PROXY"] = proxy
    os.environ["http_proxy"] = proxy
    os.environ["https_proxy"] = proxy
    os.environ["ALL_PROXY"] = f"socks5://127.0.0.1:{proxy_port}"
    os.environ["all_proxy"] = f"socks5://127.0.0.1:{proxy_port}"
    os.environ["NO_PROXY"] = "127.0.0.1,localhost"
    os.environ["no_proxy"] = "127.0.0.1,localhost"
    os.environ.pop("SKIP_MLFLOW", None)
    globals()["LOG_TO_MLFLOW"] = True

    print("Tailscale status:")
    subprocess.run(["tailscale", "status"], check=False)
    print(f"Configured HTTP/HTTPS proxy at {proxy}")
    return True


# Uncomment this in Colab after setting TS_AUTHKEY:
# setup_tailscale_for_colab()


In [5]:
BASE_FEATURE_NAMES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "vol_5d",
    "vol_20d",
    "vol_60d",
    "downside_vol_20d",
    "atr_14",
    "momentum_5d",
    "momentum_20d",
    "momentum_60d",
    "momentum_120d",
    "rsi_14",
    "macd_hist",
    "bollinger_z_20d",
    "price_to_sma_20d",
    "price_to_sma_50d",
    "price_to_sma_200d",
    "volume_zscore_20d",
    "dollar_volume_ratio_20d",
    "intraday_range",
    "beta_20d_spy",
    "beta_60d_spy",
    "excess_return_20d_vs_spy",
    "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy",
]

CUSTOM_FEATURE_NAMES = [
    "mom_vol_ratio_60d",
    "trend_spread_20_50",
    "trend_spread_50_200",
    "rolling_sharpe_60d",
    "drawdown_60d",
    "spy_vol_20d_ann",
    "spy_momentum_20d",
    "spy_momentum_60d",
    "spy_drawdown_60d",
    "market_breadth_20d",
]

ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + CUSTOM_FEATURE_NAMES


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the exact supervised LSTM-AE feature table from raw prices."""
    frame = build_features(prices, feature_names=BASE_FEATURE_NAMES)
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    eps = 1e-6

    frame["mom_vol_ratio_60d"] = frame["momentum_60d"] / (frame["vol_60d"].abs() + eps)
    frame["trend_spread_20_50"] = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["trend_spread_50_200"] = frame["price_to_sma_50d"] - frame["price_to_sma_200d"]
    frame["rolling_sharpe_60d"] = frame["return_60d"] / (frame["vol_60d"].abs() * np.sqrt(60.0) + eps)

    rolling_high_60 = panel.groupby("ticker", sort=False)["adj_close"].transform(
        lambda s: s.rolling(60, min_periods=20).max()
    )
    drawdown = panel.loc[:, ["date", "ticker"]].copy()
    drawdown["drawdown_60d"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    frame = frame.merge(drawdown, on=["date", "ticker"], how="left")

    spy = panel.loc[panel["ticker"] == "SPY", ["date", "adj_close"]].sort_values("date").copy()
    if spy.empty:
        regime = pd.DataFrame(
            {
                "date": pd.to_datetime(panel["date"].drop_duplicates()),
                "spy_vol_20d_ann": np.nan,
                "spy_momentum_20d": np.nan,
                "spy_momentum_60d": np.nan,
                "spy_drawdown_60d": np.nan,
            }
        )
    else:
        spy_ret = spy["adj_close"].pct_change()
        regime = spy.loc[:, ["date"]].copy()
        regime["spy_vol_20d_ann"] = spy_ret.rolling(20, min_periods=20).std(ddof=0) * np.sqrt(252.0)
        regime["spy_momentum_20d"] = spy["adj_close"].pct_change(20)
        regime["spy_momentum_60d"] = spy["adj_close"].pct_change(60)
        spy_high_60 = spy["adj_close"].rolling(60, min_periods=20).max()
        regime["spy_drawdown_60d"] = spy["adj_close"] / spy_high_60.replace(0.0, np.nan) - 1.0
    frame = frame.merge(regime, on="date", how="left")

    non_benchmark = panel.loc[panel["ticker"] != "SPY"].copy()
    sma_20 = non_benchmark.groupby("ticker", sort=False)["adj_close"].transform(
        lambda s: s.rolling(20, min_periods=20).mean()
    )
    breadth = (
        (non_benchmark["adj_close"] > sma_20)
        .groupby(non_benchmark["date"])
        .mean()
        .rename("market_breadth_20d")
        .reset_index()
    )
    frame = frame.merge(breadth, on="date", how="left")
    return frame.loc[:, ["date", "ticker"] + ALL_FEATURE_NAMES]


In [6]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print("Price frame shape:", prices.shape)
print("Date range:", prices["date"].min(), "->", prices["date"].max())
print("Unique tickers:", prices["ticker"].nunique())

feature_frame = build_model_features(prices)
print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())


Price frame shape: (1463605, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Unique tickers: 504
Feature frame shape: (1463605, 39)


,date,ticker,return_1d,return_5d,return_20d,return_60d,vol_5d,vol_20d,vol_60d,downside_vol_20d,...,mom_vol_ratio_60d,trend_spread_20_50,trend_spread_50_200,rolling_sharpe_60d,drawdown_60d,spy_vol_20d_ann,spy_momentum_20d,spy_momentum_60d,spy_drawdown_60d,market_breadth_20d
0,2014-01-02,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,2014-01-03,A,0.012631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,2014-01-06,A,-0.004919,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,2014-01-07,A,0.014300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,2014-01-08,A,0.016362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


## Targets And Sequences

The model predicts both forward return and benchmark-relative forward alpha. Sequences are built over the full historical panel so validation/test rows can use prior history without using future targets.


In [7]:
return_target_col = f"forward_return_{HORIZON}d"
alpha_target_col = f"forward_alpha_{HORIZON}d_vs_spy"

return_targets = make_forward_return_target(prices, horizon=HORIZON)
alpha_targets = make_forward_alpha_target(prices, horizon=HORIZON)

target_frame = feature_frame.merge(return_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(alpha_targets, on=["date", "ticker"], how="left")
target_frame = (
    target_frame.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=ALL_FEATURE_NAMES + [return_target_col, alpha_target_col])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

target_frame["historical_vol_20d"] = np.clip(target_frame["vol_20d"].to_numpy(dtype=float) * np.sqrt(252.0), 1e-4, None)

def _split_mask(frame: pd.DataFrame, start, end) -> pd.Series:
    dates = pd.to_datetime(frame["date"])
    return (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))

train_rows = target_frame.loc[_split_mask(target_frame, spec.train_start, spec.train_end)].copy()
val_rows = target_frame.loc[_split_mask(target_frame, spec.val_start, spec.val_end)].copy()
test_rows = target_frame.loc[_split_mask(target_frame, spec.test_start, spec.test_end)].copy()
train_model_rows = train_rows.loc[train_rows["ticker"] != spec.benchmark_ticker].copy()

train_means = train_model_rows[ALL_FEATURE_NAMES].mean()
train_stds = train_model_rows[ALL_FEATURE_NAMES].std(ddof=0).replace(0.0, 1.0)

target_means = {
    "return": float(train_model_rows[return_target_col].mean()),
    "alpha": float(train_model_rows[alpha_target_col].mean()),
}
target_stds = {
    "return": float(train_model_rows[return_target_col].std(ddof=0) or 1.0),
    "alpha": float(train_model_rows[alpha_target_col].std(ddof=0) or 1.0),
}

def standardize_features(frame: pd.DataFrame) -> np.ndarray:
    return ((frame[ALL_FEATURE_NAMES] - train_means) / train_stds).to_numpy(dtype=np.float32)

X_all = standardize_features(target_frame)

print("Modeling frame:", target_frame.shape)
print("Train/Val/Test rows:", len(train_rows), len(val_rows), len(test_rows))
print("Train rows excluding benchmark:", len(train_model_rows))
print("Feature count:", len(ALL_FEATURE_NAMES))
print("Target means:", target_means)
print("Target stds:", target_stds)


Modeling frame: (1360046, 42)
Train/Val/Test rows: 616108 246432 497506
Train rows excluding benchmark: 614797
Feature count: 37
Target means: {'return': 0.003379015904942063, 'alpha': 0.0007894397387451129}
Target stds: {'return': 0.03707162161352984, 'alpha': 0.03240964581331075}


In [8]:
def make_model_sequences(frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    for ticker, group in frame.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        vols = group["historical_vol_20d"].to_numpy(dtype=float)
        returns = group[return_target_col].to_numpy(dtype=float)
        alphas = group[alpha_target_col].to_numpy(dtype=float)
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            end_position = positions[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "row_position": int(end_position),
                    "target_return": float(returns[end_offset]),
                    "target_alpha": float(alphas[end_offset]),
                    "historical_vol_20d": float(vols[end_offset]),
                }
            )
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


X_seq, seq_meta = make_model_sequences(target_frame, X_all, SEQ_LEN)
seq_meta["return_scaled"] = ((seq_meta["target_return"] - target_means["return"]) / target_stds["return"]).astype(np.float32)
seq_meta["alpha_scaled"] = ((seq_meta["target_alpha"] - target_means["alpha"]) / target_stds["alpha"]).astype(np.float32)

non_benchmark_seq = seq_meta["ticker"] != spec.benchmark_ticker
train_seq_mask = _split_mask(seq_meta, spec.train_start, spec.train_end) & non_benchmark_seq
val_seq_mask = _split_mask(seq_meta, spec.val_start, spec.val_end) & non_benchmark_seq
test_seq_mask = _split_mask(seq_meta, spec.test_start, spec.test_end) & non_benchmark_seq

X_train_seq = X_seq[train_seq_mask.to_numpy()]
X_val_seq = X_seq[val_seq_mask.to_numpy()]
X_test_seq_all = X_seq[test_seq_mask.to_numpy()]

meta_train = seq_meta.loc[train_seq_mask].reset_index(drop=True)
meta_val = seq_meta.loc[val_seq_mask].reset_index(drop=True)
meta_test_all = seq_meta.loc[test_seq_mask].reset_index(drop=True)

y_train_return = meta_train["return_scaled"].to_numpy(dtype=np.float32)
y_train_alpha = meta_train["alpha_scaled"].to_numpy(dtype=np.float32)
y_val_return = meta_val["return_scaled"].to_numpy(dtype=np.float32)
y_val_alpha = meta_val["alpha_scaled"].to_numpy(dtype=np.float32)

print("All sequences:", X_seq.shape)
print("Train sequences:", X_train_seq.shape)
print("Val sequences:", X_val_seq.shape)
print("Test sequences:", X_test_seq_all.shape)


All sequences: (1350490, 20, 37)
Train sequences: (605685, 20, 37)
Val sequences: (245691, 20, 37)
Test sequences: (496319, 20, 37)


## Model And Losses

The encoder is supervised directly. Reconstruction keeps the representation grounded in the feature history, while alpha/return and rank objectives push the latent space toward stock selection.


In [9]:
class SupervisedLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int, latent_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_size = int(hidden_size)
        self.latent_dim = int(latent_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.to_latent = nn.Sequential(
            nn.Linear(hidden_size, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.Tanh(),
        )
        self.decoder_seed = nn.Linear(latent_dim, hidden_size)
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.reconstruction_head = nn.Linear(hidden_size, input_dim)
        self.return_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
        self.alpha_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        _, (h_n, _) = self.encoder(x)
        latent = self.to_latent(h_n[-1])
        repeated = self.decoder_seed(latent).unsqueeze(1).repeat(1, x.shape[1], 1)
        decoded, _ = self.decoder(repeated)
        return {
            "reconstruction": self.reconstruction_head(decoded),
            "expected_return": self.return_head(latent).squeeze(-1),
            "expected_alpha": self.alpha_head(latent).squeeze(-1),
            "latent": latent,
        }


def apply_feature_denoising(x: torch.Tensor, mask_rate: float) -> torch.Tensor:
    if mask_rate <= 0.0:
        return x
    keep = (torch.rand_like(x) > mask_rate).to(x.dtype)
    return x * keep


def pairwise_rank_loss(pred_alpha: torch.Tensor, true_alpha: torch.Tensor, date_codes: torch.Tensor) -> torch.Tensor:
    losses = []
    for date_code in torch.unique(date_codes):
        idx = torch.where(date_codes == date_code)[0]
        if idx.numel() < 2:
            continue
        pred = pred_alpha[idx]
        target = true_alpha[idx]
        target_diff = target[:, None] - target[None, :]
        useful_pairs = target_diff > 0
        if useful_pairs.any():
            pred_diff = pred[:, None] - pred[None, :]
            losses.append(F.softplus(-pred_diff[useful_pairs]).mean())
    if not losses:
        return pred_alpha.new_tensor(0.0)
    return torch.stack(losses).mean()


def batch_loss(
    model: nn.Module,
    xb: torch.Tensor,
    y_return: torch.Tensor,
    y_alpha: torch.Tensor,
    date_codes: torch.Tensor,
    *,
    denoise: bool,
) -> tuple[torch.Tensor, dict[str, float]]:
    clean_x = xb
    noisy_x = apply_feature_denoising(clean_x, MASK_RATE) if denoise else clean_x
    output = model(noisy_x)
    recon_loss = F.mse_loss(output["reconstruction"], clean_x)
    return_loss = F.mse_loss(output["expected_return"], y_return)
    alpha_loss = F.mse_loss(output["expected_alpha"], y_alpha)
    rank_loss = pairwise_rank_loss(output["expected_alpha"], y_alpha, date_codes)
    total = (
        RECON_WEIGHT * recon_loss
        + RETURN_WEIGHT * return_loss
        + ALPHA_WEIGHT * alpha_loss
        + RANK_WEIGHT * rank_loss
    )
    parts = {
        "loss": float(total.detach().cpu()),
        "recon": float(recon_loss.detach().cpu()),
        "return": float(return_loss.detach().cpu()),
        "alpha": float(alpha_loss.detach().cpu()),
        "rank": float(rank_loss.detach().cpu()),
    }
    return total, parts


def iter_date_batches(
    X: np.ndarray,
    y_return: np.ndarray,
    y_alpha: np.ndarray,
    dates: np.ndarray,
    *,
    dates_per_batch: int,
    shuffle: bool,
    rng: np.random.Generator,
):
    unique_dates = np.array(pd.unique(pd.Series(pd.to_datetime(dates))))
    if shuffle:
        rng.shuffle(unique_dates)
    for start in range(0, len(unique_dates), dates_per_batch):
        chosen_dates = unique_dates[start : start + dates_per_batch]
        idx = np.flatnonzero(np.isin(pd.to_datetime(dates).to_numpy(), chosen_dates))
        if shuffle:
            rng.shuffle(idx)
        date_codes = pd.factorize(pd.to_datetime(dates[idx]))[0].astype(np.int64)
        yield X[idx], y_return[idx], y_alpha[idx], date_codes


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SupervisedLSTMAutoencoder(
    input_dim=len(ALL_FEATURE_NAMES),
    hidden_size=HIDDEN_SIZE,
    latent_dim=LATENT_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

train_dates = meta_train["date"].to_numpy(dtype="datetime64[ns]")
val_dates = meta_val["date"].to_numpy(dtype="datetime64[ns]")


def evaluate_losses(model: nn.Module, X: np.ndarray, y_return: np.ndarray, y_alpha: np.ndarray, dates: np.ndarray) -> dict[str, float]:
    model.eval()
    rng = np.random.default_rng(RANDOM_SEED)
    totals = {"loss": 0.0, "recon": 0.0, "return": 0.0, "alpha": 0.0, "rank": 0.0, "n": 0.0}
    with torch.no_grad():
        for xb_np, yr_np, ya_np, date_codes_np in iter_date_batches(
            X,
            y_return,
            y_alpha,
            dates,
            dates_per_batch=DATES_PER_BATCH,
            shuffle=False,
            rng=rng,
        ):
            xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
            yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
            ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
            dc = torch.as_tensor(date_codes_np, dtype=torch.long, device=device)
            _, parts = batch_loss(model, xb, yr, ya, dc, denoise=False)
            batch_n = float(len(xb_np))
            for key in ["loss", "recon", "return", "alpha", "rank"]:
                totals[key] += parts[key] * batch_n
            totals["n"] += batch_n
    return {key: totals[key] / max(totals["n"], 1.0) for key in ["loss", "recon", "return", "alpha", "rank"]}


history = []
best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
best_val_loss = float("inf")
patience_counter = 0
rng = np.random.default_rng(RANDOM_SEED)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_totals = {"loss": 0.0, "recon": 0.0, "return": 0.0, "alpha": 0.0, "rank": 0.0, "n": 0.0}
    for xb_np, yr_np, ya_np, date_codes_np in iter_date_batches(
        X_train_seq,
        y_train_return,
        y_train_alpha,
        train_dates,
        dates_per_batch=DATES_PER_BATCH,
        shuffle=True,
        rng=rng,
    ):
        xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
        yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
        ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
        dc = torch.as_tensor(date_codes_np, dtype=torch.long, device=device)

        optimizer.zero_grad(set_to_none=True)
        loss, parts = batch_loss(model, xb, yr, ya, dc, denoise=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        batch_n = float(len(xb_np))
        for key in ["loss", "recon", "return", "alpha", "rank"]:
            train_totals[key] += parts[key] * batch_n
        train_totals["n"] += batch_n

    train_metrics = {key: train_totals[key] / max(train_totals["n"], 1.0) for key in ["loss", "recon", "return", "alpha", "rank"]}
    val_metrics = evaluate_losses(model, X_val_seq, y_val_return, y_val_alpha, val_dates)
    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print(
        f"epoch {epoch:02d}/{EPOCHS} | train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | val alpha {val_metrics['alpha']:.4f} | val rank {val_metrics['rank']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss - 1e-5:
        best_val_loss = val_metrics["loss"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(best_state)
history = pd.DataFrame(history)
display(history.tail())
print("Best validation combined loss:", best_val_loss)


epoch 01/18 | train loss 3.3540 | val loss 7.4021 | val alpha 2.1166 | val rank 0.6931
epoch 02/18 | train loss 3.1074 | val loss 7.1452 | val alpha 2.1163 | val rank 0.6934
epoch 03/18 | train loss 3.0423 | val loss 7.0457 | val alpha 2.1231 | val rank 0.6943
epoch 04/18 | train loss 2.9990 | val loss 6.9379 | val alpha 2.1160 | val rank 0.6943
epoch 05/18 | train loss 2.9759 | val loss 6.9419 | val alpha 2.1271 | val rank 0.6956
epoch 06/18 | train loss 2.9555 | val loss 6.9362 | val alpha 2.1342 | val rank 0.6954
epoch 07/18 | train loss 2.9405 | val loss 6.8645 | val alpha 2.1298 | val rank 0.6990
epoch 08/18 | train loss 2.9204 | val loss 6.9327 | val alpha 2.1596 | val rank 0.7001
epoch 09/18 | train loss 2.9091 | val loss 6.9546 | val alpha 2.1736 | val rank 0.7059
epoch 10/18 | train loss 2.8898 | val loss 6.8920 | val alpha 2.1515 | val rank 0.7035
epoch 11/18 | train loss 2.8773 | val loss 7.1731 | val alpha 2.2632 | val rank 0.7219
epoch 12/18 | train loss 2.8687 | val loss 

,epoch,train_loss,train_recon,train_return,train_alpha,train_rank,val_loss,val_recon,val_return,val_alpha,val_rank
7,8,2.920370,0.320604,0.976477,0.986686,0.690775,6.932650,1.221818,2.503110,2.159626,0.700130
8,9,2.909068,0.314547,0.972928,0.984979,0.690491,6.954577,1.206057,2.520340,2.173585,0.705899
9,10,2.889791,0.308922,0.966625,0.979786,0.689925,6.891960,1.185888,2.524629,2.151530,0.703493
10,11,2.877327,0.305314,0.962179,0.976526,0.689352,7.173113,1.215824,2.573181,2.263164,0.721853
11,12,2.868651,0.303765,0.956890,0.974315,0.689057,7.038732,1.180557,2.587752,2.210639,0.715108


Best validation combined loss: 6.864469216559091


## Diagnostics

These diagnostics check whether the model is learning cross-sectional ordering before the backtest is trusted.


In [11]:
def predict_scaled(model: nn.Module, X: np.ndarray, batch_size: int = 4096) -> pd.DataFrame:
    model.eval()
    returns, alphas, latent_norms = [], [], []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32, device=device)
            output = model(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())
    return pd.DataFrame(
        {
            "return_scaled_pred": np.concatenate(returns),
            "alpha_scaled_pred": np.concatenate(alphas),
            "latent_norm": np.concatenate(latent_norms),
        }
    )


def unscale_predictions(predicted: pd.DataFrame) -> pd.DataFrame:
    result = predicted.copy()
    result["expected_return"] = result["return_scaled_pred"] * target_stds["return"] + target_means["return"]
    result["expected_alpha"] = result["alpha_scaled_pred"] * target_stds["alpha"] + target_means["alpha"]
    return result


def rank_ic_by_year(frame: pd.DataFrame, score_col: str, target_col: str) -> pd.DataFrame:
    daily_rows = []
    for date_value, group in frame.groupby("date", sort=True):
        if group[score_col].nunique() < 2 or group[target_col].nunique() < 2:
            continue
        daily_rows.append(
            {
                "date": pd.Timestamp(date_value),
                "rank_ic": group[score_col].rank().corr(group[target_col].rank()),
            }
        )
    if not daily_rows:
        return pd.DataFrame(columns=["year", "mean_rank_ic", "observations"])
    daily = pd.DataFrame(daily_rows)
    daily["year"] = daily["date"].dt.year
    return daily.groupby("year").agg(mean_rank_ic=("rank_ic", "mean"), observations=("rank_ic", "count")).reset_index()


def top_bottom_spread_by_year(frame: pd.DataFrame, score_col: str, target_col: str, buckets: int = 5) -> pd.DataFrame:
    rows = []
    for date_value, group in frame.groupby("date", sort=True):
        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        if len(group) < buckets:
            continue
        bucket_size = max(1, len(group) // buckets)
        top = group.head(bucket_size)[target_col].mean()
        bottom = group.tail(bucket_size)[target_col].mean()
        rows.append({"date": pd.Timestamp(date_value), "top_minus_bottom_alpha": float(top - bottom)})
    if not rows:
        return pd.DataFrame(columns=["year", "mean_top_minus_bottom_alpha", "observations"])
    spreads = pd.DataFrame(rows)
    spreads["year"] = spreads["date"].dt.year
    return spreads.groupby("year").agg(
        mean_top_minus_bottom_alpha=("top_minus_bottom_alpha", "mean"),
        observations=("top_minus_bottom_alpha", "count"),
    ).reset_index()


val_pred_scaled = unscale_predictions(predict_scaled(model, X_val_seq))
val_diagnostics = pd.concat([meta_val.copy(), val_pred_scaled], axis=1)
val_return_mse = float(np.mean((val_diagnostics["expected_return"] - val_diagnostics["target_return"]) ** 2))
val_alpha_mse = float(np.mean((val_diagnostics["expected_alpha"] - val_diagnostics["target_alpha"]) ** 2))

print("Validation return MSE:", val_return_mse)
print("Validation alpha MSE:", val_alpha_mse)
print("Rank IC by year:")
display(rank_ic_by_year(val_diagnostics, "expected_alpha", "target_alpha"))
print("Top-minus-bottom alpha spread by year:")
display(top_bottom_spread_by_year(val_diagnostics, "expected_alpha", "target_alpha"))


Validation return MSE: 0.0034350707439934056
Validation alpha MSE: 0.002237066320493641
Rank IC by year:


,year,mean_rank_ic,observations
0,2020,0.006563,253
1,2021,0.024032,252


Top-minus-bottom alpha spread by year:


,year,mean_top_minus_bottom_alpha,observations
0,2020,0.005512,253
1,2021,0.004008,252


## Predictions And Weekly Portfolio

Portfolio construction uses `expected_alpha / historical_vol_20d`, rank-normalized by date. The backtest rebalances weekly because the weights table contains only weekly rows.


In [12]:
def rank_normalize_by_date(frame: pd.DataFrame, score_col: str, out_col: str) -> pd.DataFrame:
    result = frame.copy()
    pct_rank = result.groupby("date")[score_col].rank(method="average", pct=True)
    result[out_col] = 2.0 * pct_rank - 1.0
    return result


def select_rebalance_dates(dates: pd.Series | pd.DatetimeIndex, frequency: str = "weekly") -> pd.DatetimeIndex:
    unique_dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates).drop_duplicates()).sort_values())
    if frequency == "daily":
        return unique_dates
    date_series = pd.Series(unique_dates, index=unique_dates)
    if frequency == "weekly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("W-FRI")).max().to_numpy())
    if frequency == "monthly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("M")).max().to_numpy())
    raise ValueError("frequency must be one of: daily, weekly, monthly")


def apply_weight_cap(raw_weights: pd.Series, max_weight: float) -> pd.Series:
    raw = raw_weights.clip(lower=0.0).astype(float)
    raw = raw.loc[raw > 0.0]
    if raw.empty:
        return raw_weights * 0.0
    weights = pd.Series(0.0, index=raw.index, dtype=float)
    remaining = raw.copy()
    budget = 1.0
    while not remaining.empty:
        scaled = remaining / remaining.sum() * budget
        over_cap = scaled > max_weight
        if not over_cap.any():
            weights.loc[remaining.index] = scaled
            break
        capped_names = scaled.loc[over_cap].index
        weights.loc[capped_names] = max_weight
        budget -= max_weight * len(capped_names)
        remaining = remaining.drop(index=capped_names)
        if budget <= 1e-12:
            break
    total = weights.sum()
    if total <= 0.0:
        weights = pd.Series(1.0 / len(raw), index=raw.index, dtype=float)
    elif abs(total - 1.0) > 1e-10:
        weights = weights / total
    return weights


def weights_from_rank_scores_weekly(
    predictions: pd.DataFrame,
    *,
    score_col: str,
    dataset_name: str,
    strategy_name: str,
    max_weight: float,
    top_fraction: float,
    turnover_blend: float,
    universe_tickers: list[str] | None = None,
) -> PortfolioWeights:
    tickers = sorted(predictions["ticker"].unique()) if universe_tickers is None else list(universe_tickers)
    rows = []
    previous = None
    min_names = int(math.ceil(1.0 / max_weight))
    for date_value, group in predictions.groupby("date", sort=True):
        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        top_n = min(len(group), max(min_names, int(math.ceil(len(group) * top_fraction))))
        selected = group.head(top_n).copy()
        raw_scores = selected[score_col] - selected[score_col].min() + 1e-6
        raw = pd.Series(raw_scores.to_numpy(dtype=float), index=selected["ticker"])
        raw = raw.loc[raw.index.isin(tickers)]
        capped = apply_weight_cap(raw, max_weight=max_weight)
        row = pd.Series(0.0, index=tickers, dtype=float)
        row.loc[capped.index] = capped.to_numpy(dtype=float)
        if row.sum() <= 0.0:
            fallback = [ticker for ticker in group["ticker"] if ticker in tickers]
            if not fallback:
                fallback = tickers
            row.loc[fallback] = 1.0 / len(fallback)
        if previous is not None:
            row = turnover_blend * row + (1.0 - turnover_blend) * previous
            row = row / row.sum()
        row.name = pd.Timestamp(date_value)
        rows.append(row)
        previous = row
    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    return PortfolioWeights(
        weights=weights,
        dataset_name=dataset_name,
        strategy_name=strategy_name,
        metadata={
            "score_col": score_col,
            "rebalance_frequency": REBALANCE_FREQUENCY,
            "max_weight": max_weight,
            "top_fraction": top_fraction,
            "turnover_blend": turnover_blend,
            "universe_ticker_count": len(tickers),
        },
    )


test_pred_scaled = unscale_predictions(predict_scaled(model, X_test_seq_all))
test_scored_all = pd.concat([meta_test_all.copy(), test_pred_scaled], axis=1)
test_scored_all["expected_volatility"] = np.clip(test_scored_all["historical_vol_20d"], 1e-4, None)
test_scored_all["alpha_vol_score"] = test_scored_all["expected_alpha"] / test_scored_all["expected_volatility"]
test_scored_all = rank_normalize_by_date(test_scored_all, "alpha_vol_score", "rank_normalized_score")
test_scored_all["uncertainty"] = (test_scored_all["expected_return"] - test_scored_all["expected_alpha"]).abs()

weekly_dates = select_rebalance_dates(test_scored_all["date"], REBALANCE_FREQUENCY)
weekly_scored = test_scored_all.loc[test_scored_all["date"].isin(weekly_dates)].copy()

predictions = weekly_scored.loc[
    :,
    [
        "date",
        "ticker",
        "expected_return",
        "expected_alpha",
        "expected_volatility",
        "uncertainty",
        "alpha_vol_score",
        "rank_normalized_score",
    ],
].copy()
predictions["horizon"] = HORIZON
predictions = predictions.loc[
    :,
    [
        "date",
        "ticker",
        "horizon",
        "expected_return",
        "expected_alpha",
        "expected_volatility",
        "uncertainty",
        "alpha_vol_score",
        "rank_normalized_score",
    ],
]

predictions = validate_prediction_frame(
    predictions,
    dataset_name=DATASET_NAME,
    horizon=HORIZON,
    repo_root=repo_root,
)

portfolio = weights_from_rank_scores_weekly(
    predictions,
    score_col="rank_normalized_score",
    dataset_name=DATASET_NAME,
    strategy_name=MODEL_NAME,
    max_weight=MAX_WEIGHT,
    top_fraction=TOP_FRACTION,
    turnover_blend=TURNOVER_BLEND,
    universe_tickers=tradable_tickers,
)
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root)

print("Predictions:", predictions.shape)
print("Weekly weight rows:", validated_weights.shape)
print("Prediction date range:", predictions["date"].min(), "->", predictions["date"].max())
display(predictions.head())
display(validated_weights.head())


Predictions: (103447, 9)
Weekly weight rows: (208, 503)
Prediction date range: 2022-01-07 00:00:00 -> 2025-12-23 00:00:00


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,uncertainty,alpha_vol_score,rank_normalized_score
0,2022-01-07,A,5,0.006238,0.001475,0.255771,0.004763,0.005767,0.556911
1,2022-01-07,AAPL,5,0.003076,0.000678,0.291345,0.002398,0.002326,0.191057
2,2022-01-07,ABBV,5,-0.001849,-0.009476,0.139997,0.007626,-0.067684,-0.987805
3,2022-01-07,ABNB,5,0.006845,0.006053,0.471048,0.000791,0.012851,0.853659
4,2022-01-07,ABT,5,0.003454,-0.000386,0.205231,0.003840,-0.001879,-0.268293


,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,WMB,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS
date,,,,,,,,,,,,,,,,,,,,,
2022-01-07,0.002854,0.001891,0.000000,0.0,8.163249e-09,0.003219,0.007865,0.003883,0.001029,0.004513,...,0.004845,0.0,0.006869,0.000000,0.0,0.005077,0.000000,0.001294,0.000000,0.006006
2022-01-14,0.001427,0.003816,0.001493,0.0,7.632388e-04,0.003468,0.007898,0.002970,0.000514,0.002837,...,0.002422,0.0,0.004480,0.000000,0.0,0.003849,0.000614,0.001327,0.000000,0.004480
2022-01-21,0.000713,0.005392,0.002704,0.0,3.816194e-04,0.001850,0.007931,0.001983,0.000257,0.001419,...,0.001211,0.0,0.002721,0.000000,0.0,0.001925,0.001369,0.001112,0.000000,0.003235
2022-01-28,0.000357,0.005799,0.002846,0.0,1.908097e-04,0.000925,0.007981,0.002584,0.000129,0.001390,...,0.000606,0.0,0.001526,0.003435,0.0,0.000962,0.003107,0.001020,0.001609,0.002812
2022-02-04,0.000178,0.002899,0.002468,0.0,1.190479e-03,0.000463,0.006728,0.004544,0.000064,0.004362,...,0.000303,0.0,0.000763,0.005550,0.0,0.003368,0.004407,0.002734,0.004239,0.003547


## Backtest, Artifacts, And Baseline Comparison


In [13]:
def write_artifacts_for_notebook(result, output_dir: Path) -> dict[str, str]:
    if os.environ.get("SKIP_QUANTSTATS", "0") != "1":
        return write_backtest_artifacts(result, output_dir)

    output_path = Path(output_dir).resolve()
    output_path.mkdir(parents=True, exist_ok=True)
    paths = {
        "weights": output_path / "weights.parquet",
        "nav": output_path / "nav.parquet",
        "returns": output_path / "returns.parquet",
        "turnover": output_path / "turnover.parquet",
        "benchmarks": output_path / "benchmarks.parquet",
        "metrics": output_path / "metrics.json",
        "metrics_table": output_path / "metrics_table.parquet",
        "quantstats_report": output_path / "quantstats.html",
    }
    result.weights.to_parquet(paths["weights"])
    result.nav.to_frame(name="nav").to_parquet(paths["nav"])
    result.returns.to_frame(name="returns").to_parquet(paths["returns"])
    result.turnover.to_frame(name="turnover").to_parquet(paths["turnover"])
    result.benchmark_returns.to_parquet(paths["benchmarks"])
    paths["metrics"].write_text(json.dumps(result.metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(result.metrics.items())]).to_parquet(
        paths["metrics_table"], index=False
    )
    paths["quantstats_report"].write_text(
        "<html><body><p>QuantStats skipped because SKIP_QUANTSTATS=1.</p></body></html>\n",
        encoding="utf-8",
    )
    artifact_paths = {key: str(value) for key, value in paths.items()}
    result.artifact_paths.update(artifact_paths)
    return artifact_paths


print("Running weekly backtest...")
result = backtest_weights(backtest_spec, portfolio, benchmark=spec.benchmark_ticker, repo_root=repo_root)
metrics = build_metrics(result)
result.metrics = metrics
print("Writing backtest artifacts...")
artifact_paths = write_artifacts_for_notebook(result, output_dir)

metrics_table = pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(metrics.items())])
display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])

baseline_metrics_path = repo_root / "runs" / "autoencoder_downstream" / "metrics.json"
if baseline_metrics_path.exists():
    baseline_metrics = json.loads(baseline_metrics_path.read_text())
    comparison = pd.DataFrame(
        [
            {
                "metric": metric,
                "supervised_lstm_ae": metrics.get(metric),
                "baseline_autoencoder": baseline_metrics.get(metric),
                "delta": metrics.get(metric, np.nan) - baseline_metrics.get(metric, np.nan),
            }
            for metric in ["annual_return", "sharpe", "max_drawdown", "average_turnover"]
        ]
    )
    print("Comparison against runs/autoencoder_downstream/metrics.json:")
    display(comparison)
else:
    print("No existing baseline metrics found at", baseline_metrics_path)


Running weekly backtest...
Writing backtest artifacts...


,metric,value
0,annual_excess_return_vs_benchmark,0.038198
1,annual_return,0.153702
2,annual_volatility,0.231769
3,average_turnover,0.162201
4,benchmark_annual_return,0.115504
5,benchmark_annual_volatility,0.179651
6,benchmark_max_drawdown,-0.234240
7,benchmark_sharpe,0.642936
8,benchmark_total_return,0.545167
9,calmar,0.571983


QuantStats report: /content/Portfolio-Optimization-Lib/runs/supervised_lstm_autoencoder/quantstats.html
No existing baseline metrics found at /content/Portfolio-Optimization-Lib/runs/autoencoder_downstream/metrics.json


## Reusable Inference And Checkpoint

`predict_from_prices` is the hidden-subset style entrypoint. It rebuilds features from raw prices, constructs ticker sequences, runs the trained model, and returns the standard prediction frame.


In [14]:
def make_inference_sequences(feature_frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    working = feature_frame.sort_values(["ticker", "date"]).reset_index(drop=True)
    for ticker, group in working.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        vols = np.clip(group["vol_20d"].to_numpy(dtype=float) * np.sqrt(252.0), 1e-4, None)
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "historical_vol_20d": float(vols[end_offset]),
                }
            )
    if not Xs:
        return np.empty((0, seq_len, len(ALL_FEATURE_NAMES)), dtype=np.float32), pd.DataFrame(meta)
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


def _series_from_mapping(value) -> pd.Series:
    return value if isinstance(value, pd.Series) else pd.Series(value, dtype=float)


def predict_from_prices(model_bundle: dict, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    model_obj = model_bundle["model"]
    model_device = model_bundle.get("device", device)
    feature_names = list(model_bundle["feature_names"])
    seq_len = int(model_bundle["seq_len"])
    horizon = int(model_bundle["horizon"])
    train_mean = _series_from_mapping(model_bundle["train_means"])
    train_std = _series_from_mapping(model_bundle["train_stds"])
    t_means = dict(model_bundle["target_means"])
    t_stds = dict(model_bundle["target_stds"])

    features = build_model_features(prices).replace([np.inf, -np.inf], np.nan)
    if tickers is not None:
        tickers_normalized = [str(ticker).upper() for ticker in tickers]
        features = features.loc[features["ticker"].isin(tickers_normalized)].copy()
    features = features.dropna(subset=feature_names).sort_values(["ticker", "date"]).reset_index(drop=True)
    if features.empty:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    X = ((features[feature_names] - train_mean.loc[feature_names]) / train_std.loc[feature_names]).to_numpy(dtype=np.float32)
    X_infer, meta = make_inference_sequences(features, X, seq_len)
    if len(meta) == 0:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    model_obj.eval()
    returns, alphas, latent_norms = [], [], []
    with torch.no_grad():
        for start in range(0, len(X_infer), 4096):
            xb = torch.as_tensor(X_infer[start : start + 4096], dtype=torch.float32, device=model_device)
            output = model_obj(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())

    scored = meta.copy()
    scored["expected_return"] = np.concatenate(returns) * t_stds["return"] + t_means["return"]
    scored["expected_alpha"] = np.concatenate(alphas) * t_stds["alpha"] + t_means["alpha"]
    scored["expected_volatility"] = np.clip(scored["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    scored["uncertainty"] = np.abs(scored["expected_return"] - scored["expected_alpha"])
    scored["alpha_vol_score"] = scored["expected_alpha"] / scored["expected_volatility"]
    scored = rank_normalize_by_date(scored, "alpha_vol_score", "rank_normalized_score")
    scored["horizon"] = horizon

    if dates is not None:
        requested_dates = pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None)
        scored = scored.loc[scored["date"].isin(set(requested_dates))].copy()

    return scored.loc[
        :,
        [
            "date",
            "ticker",
            "horizon",
            "expected_return",
            "expected_alpha",
            "expected_volatility",
            "uncertainty",
            "alpha_vol_score",
            "rank_normalized_score",
        ],
    ].reset_index(drop=True)


model_config = {
    "input_dim": len(ALL_FEATURE_NAMES),
    "hidden_size": HIDDEN_SIZE,
    "latent_dim": LATENT_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
}

checkpoint = {
    "model_state": model.state_dict(),
    "model_config": model_config,
    "feature_names": ALL_FEATURE_NAMES,
    "base_feature_names": BASE_FEATURE_NAMES,
    "custom_feature_names": CUSTOM_FEATURE_NAMES,
    "train_means": train_means.to_dict(),
    "train_stds": train_stds.to_dict(),
    "target_means": target_means,
    "target_stds": target_stds,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "random_seed": RANDOM_SEED,
}
model_artifact_path = output_dir / "supervised_lstm_autoencoder.pt"
torch.save(checkpoint, model_artifact_path)
print("Saved model checkpoint:", model_artifact_path)

model_bundle = {
    "model": model,
    "device": device,
    "feature_names": ALL_FEATURE_NAMES,
    "train_means": train_means,
    "train_stds": train_stds,
    "target_means": target_means,
    "target_stds": target_stds,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
}

# Smoke-check the reusable inference path on the same weekly test dates.
submission_style_predictions = predict_from_prices(model_bundle, prices, dates=weekly_dates, tickers=validated_weights.columns)
submission_style_predictions = validate_prediction_frame(
    submission_style_predictions,
    dataset_name=DATASET_NAME,
    horizon=HORIZON,
    repo_root=repo_root,
)
print("Reusable inference predictions:", submission_style_predictions.shape)
display(submission_style_predictions.head())


Saved model checkpoint: /content/Portfolio-Optimization-Lib/runs/supervised_lstm_autoencoder/supervised_lstm_autoencoder.pt
Reusable inference predictions: (103447, 9)


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,uncertainty,alpha_vol_score,rank_normalized_score
0,2022-01-07,A,5,0.006238,0.001475,0.255771,0.004763,0.005767,0.556911
1,2022-01-07,AAPL,5,0.003076,0.000678,0.291345,0.002398,0.002326,0.191057
2,2022-01-07,ABBV,5,-0.001849,-0.009476,0.139997,0.007626,-0.067684,-0.987805
3,2022-01-07,ABNB,5,0.006845,0.006053,0.471048,0.000791,0.012851,0.853659
4,2022-01-07,ABT,5,0.003454,-0.000386,0.205231,0.003840,-0.001879,-0.268293


## MLflow Logging

This logs the normal research outputs and a reconstructable model-submission bundle when the configured MLflow server is reachable. Set `SKIP_MLFLOW=1` for local smoke execution.


In [15]:
def mlflow_tracking_preflight(tracking_uri: str, timeout: float = 2.0) -> tuple[bool, str]:
    """Return quickly before MLflow starts urllib retry loops in disconnected runtimes."""
    from urllib.error import HTTPError, URLError
    from urllib.parse import urlparse
    from urllib.request import Request, urlopen
    import socket

    parsed = urlparse(tracking_uri)
    if parsed.scheme not in {"http", "https"}:
        return True, "local tracking URI"
    if not parsed.hostname:
        return False, f"remote tracking URI has no hostname: {tracking_uri}"

    proxy_env = ["HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "https_proxy", "http_proxy", "all_proxy"]
    configured_proxy = next((name for name in proxy_env if os.environ.get(name)), None)
    if configured_proxy:
        try:
            request = Request(tracking_uri.rstrip("/") + "/", method="GET")
            with urlopen(request, timeout=timeout):
                return True, f"{parsed.hostname} reachable through {configured_proxy}"
        except HTTPError as exc:
            # Any HTTP response means DNS/routing reached the server or proxy.
            return True, f"{parsed.hostname} reachable through {configured_proxy}; HTTP {exc.code}"
        except URLError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc.reason})"
        except OSError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc})"

    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((parsed.hostname, port), timeout=timeout):
            return True, f"{parsed.hostname}:{port} reachable"
    except OSError as exc:
        return False, f"{parsed.hostname}:{port} not reachable ({exc})"


if LOG_TO_MLFLOW:
    mlflow_layout = init_mlflow(repo_root)
    tracking_uri = mlflow_layout["tracking_uri"]
    print("MLflow tracking URI:", tracking_uri)
    tracking_ok, tracking_reason = mlflow_tracking_preflight(tracking_uri)
    if not tracking_ok:
        print("MLflow logging skipped:", tracking_reason)
        print("Set SKIP_MLFLOW=1 to suppress this cell, or connect to the MLflow/Tailscale host and rerun it.")
    else:
        try:
            with start_run(
                run_name=RUN_NAME,
                dataset_name=DATASET_NAME,
                tags={
                    "workflow": "supervised_lstm_autoencoder",
                    "model_family": "torch_lstm_autoencoder",
                    "prediction_horizon": str(HORIZON),
                    "rebalance_frequency": REBALANCE_FREQUENCY,
                },
                repo_root=repo_root,
            ):
                import mlflow

                mlflow.log_params(
                    {
                        "model_name": MODEL_NAME,
                        "dataset_name": DATASET_NAME,
                        "horizon": HORIZON,
                        "seq_len": SEQ_LEN,
                        "feature_count": len(ALL_FEATURE_NAMES),
                        "latent_dim": LATENT_DIM,
                        "hidden_size": HIDDEN_SIZE,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                        "learning_rate": LEARNING_RATE,
                        "weight_decay": WEIGHT_DECAY,
                        "epochs_requested": EPOCHS,
                        "epochs_run": len(history),
                        "mask_rate": MASK_RATE,
                        "loss_weights": json.dumps(
                            {
                                "reconstruction": RECON_WEIGHT,
                                "return": RETURN_WEIGHT,
                                "alpha": ALPHA_WEIGHT,
                                "rank": RANK_WEIGHT,
                            },
                            sort_keys=True,
                        ),
                        "portfolio_score": "expected_alpha / historical_vol_20d_ann",
                        "rank_normalized_by_date": True,
                        "portfolio_top_fraction": TOP_FRACTION,
                        "portfolio_max_weight": MAX_WEIGHT,
                        "portfolio_turnover_blend": TURNOVER_BLEND,
                        "cost_bps": spec.cost_bps,
                        "random_seed": RANDOM_SEED,
                        "val_return_mse": val_return_mse,
                        "val_alpha_mse": val_alpha_mse,
                    }
                )
                log_predictions(predictions)
                log_portfolio(portfolio)
                log_backtest(result)
                manifest = log_model_submission(
                    {"model_checkpoint": model_artifact_path},
                    model_name=MODEL_NAME,
                    model_family="torch",
                    feature_names=ALL_FEATURE_NAMES,
                    target=alpha_target_col,
                    horizon=HORIZON,
                    rebalance_frequency=REBALANCE_FREQUENCY,
                    preprocessing={
                        "scaler": "train_mean_std_non_benchmark_rows",
                        "target_scaler": "train_target_mean_std_non_benchmark_rows",
                        "train_means": train_means.to_dict(),
                        "train_stds": train_stds.to_dict(),
                        "target_means": target_means,
                        "target_stds": target_stds,
                    },
                    model_config={
                        **model_config,
                        "seq_len": SEQ_LEN,
                        "mask_rate": MASK_RATE,
                        "loss_weights": {
                            "reconstruction": RECON_WEIGHT,
                            "return": RETURN_WEIGHT,
                            "alpha": ALPHA_WEIGHT,
                            "rank": RANK_WEIGHT,
                        },
                        "portfolio": portfolio.metadata,
                    },
                    source_files=[NOTEBOOK_PATH] if NOTEBOOK_PATH.exists() else None,
                    notes="Supervised denoising LSTM autoencoder with weekly alpha/vol rank portfolio.",
                )
                print("MLflow logging complete. Submission manifest keys:", sorted(manifest.keys()))
        except Exception as exc:
            print("MLflow logging skipped after error:", repr(exc))
else:
    print("MLflow logging skipped because SKIP_MLFLOW=1.")


MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
MLflow logging skipped: adams-macbook-pro.tail5ddc35.ts.net:443 not reachable ([Errno -2] Name or service not known)
Set SKIP_MLFLOW=1 to suppress this cell, or connect to the MLflow/Tailscale host and rerun it.


## Final Checks


In [16]:
assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert set(validated_weights.columns).isdisjoint({spec.benchmark_ticker}), "Benchmark ticker should not be traded."
assert validated_weights.index.is_monotonic_increasing
assert pd.Index(validated_weights.index.to_period("W-FRI")).is_unique, "Weights should contain one row per week."
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert float(validated_weights.max(axis=1).max()) <= MAX_WEIGHT + 1e-6
assert predictions["expected_volatility"].notna().all()
assert predictions["rank_normalized_score"].notna().all()
assert predictions["ticker"].ne(spec.benchmark_ticker).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert model_artifact_path.exists()
assert len(submission_style_predictions) == len(predictions)

print("End-to-end supervised LSTM autoencoder workflow validated successfully.")
print("Key metrics:")
for key in ["annual_return", "sharpe", "max_drawdown", "average_turnover"]:
    print(f"  {key:<20s}: {result.metrics[key]:.6f}")

display(result.nav.tail().to_frame("nav"))
display(result.turnover.tail().to_frame("turnover"))


End-to-end supervised LSTM autoencoder workflow validated successfully.
Key metrics:
  annual_return       : 0.153702
  sharpe              : 0.663171
  max_drawdown        : -0.268718
  average_turnover    : 0.162201


,nav
2025-12-24,178.568034
2025-12-26,178.540116
2025-12-29,178.231222
2025-12-30,178.088100
2025-12-31,176.502533


,turnover
date,
2025-11-28,0.146592
2025-12-05,0.161078
2025-12-12,0.175443
2025-12-19,0.150423
2025-12-23,0.111923
